### Эксперимент 2. Плучение эмбеддингов для Qwen

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()


In [ ]:
verb = "малтать"
definition = "молча кивать, когда ничего не понял"

# prompt = f'''В русском появился новый глагол "{verb}", который значит "{definition}".
# Как одним словом назвать человека, который {verb}?
# Ответь только одним существительным.'''

prompt = f'''Есть глагол {verb}. Как одним словом c тем же корнем как у {verb} назвать человека, который это делает? Ответь только одним существительным.'''

messages = [{"role": "user", "content": prompt}]
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    gen = model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(
    gen[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("Derivation:", answer)

In [ ]:
import numpy as np
import os
import torch
from pathlib import Path

In [ ]:
# Создаем директорию для сохранения эмбеддингов
EMBEDDINGS_DIR = "embeddings_npy"
Path(EMBEDDINGS_DIR).mkdir(parents=True, exist_ok=True)

def mean_pool_last_hidden(text: str):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(
            **enc,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )

    last_hidden = out.hidden_states[-1]
    mask = enc["attention_mask"].unsqueeze(-1)

    pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return pooled[0].detach().cpu()

def save_embedding(embedding, verb, definition):
    """Сохраняем эмбеддинг в .npy файл"""


    safe_verb = verb.replace('/', '_').replace('\\', '_').replace(' ', '_')
    npy_path = os.path.join(EMBEDDINGS_DIR, f"{safe_verb}.npy")


    if embedding.dtype == torch.bfloat16:
        embedding = embedding.float()


    np.save(npy_path, embedding.numpy())


    print(f"   Файл: {npy_path}")
    print(f"   Размерность: {embedding.shape}")
    print(f"   Тип данных: {embedding.dtype}")

    return npy_path


    print(f"   Файл: {npy_path}")
    print(f"   Размерность: {embedding.shape}")
    print(f"   Тип данных: {embedding.dtype}")

    return npy_path

embedding = mean_pool_last_hidden(definition)


print(f"   Форма эмбеддинга: {embedding.shape}")
print(f"   Первые 10 значений: {embedding[:10]}")
print(f"   Последние 10 значений: {embedding[-10:]}")

save_embedding(embedding, verb, definition)